# 10 · X3: the final retrieval pipeline (phase A: GPU encoder)

X3 = BM25 + dense → RRF (k = 10) → keyword engine-type filter → 30 candidates → jina-reranker-v3 → score cut-off ≥ 0.0, keep 1–8 (params `v3`, windows index `cse_win500_v1`). The notebook runs X3 on the questions of the active experiment, measures it like every other run, compares it with X2, and saves the retrieved contexts for the generation notebooks. Design and numbers: `x2_to_x3_plan.md`.

> **⚠ STOP vLLM BEFORE RUNNING THIS NOTEBOOK.** Qdrant and the **GPU encoder** (`encoder-gpu`) must be running, and the GPU encoder and vLLM do not fit on the card together.
> ```bash
> docker compose stop vllm encoder                   # vLLM off; the CPU encoder shares port 8081
> docker compose up -d qdrant && docker compose up -d --force-recreate encoder-gpu
> ```
> The guard cell below refuses to run while vLLM answers on port 8000, and refuses to run on a CPU encoder unless you set `ALLOW_CPU_RERANK = True`.
>
> **After this notebook**, the generation notebooks need only vLLM, because they read `results/<experiment>/x3/contexts.jsonl` and use neither Qdrant nor the encoder:
> ```bash
> docker compose stop encoder-gpu && docker compose up -d vllm      # VLLM_CONFIG=serve.retrieval.yaml (.env); ~2.5 min to healthy
> ```

| step | what |
|---|---|
| 1 | parameters, the keyword rules, the X3 config, the baselines (the stored v1 runs X2 gate off and on, H1) |
| 2 | the keyword rule against the labels; **the gold sections the rule would remove** |
| 3 | run X3 (resumable, cached) |
| 4 | the standard metric files of notebook 09 for X3 and the baselines, and the pipeline diagram |
| 5 | the X2 vs X3 table, by family, paired differences, the questions where X3 loses evidence |
| 6 | other cut-offs on X3's own reranked lists (no reranker call) |
| 7 | the unanswerable questions (inspection only) |
| 8 | the hand-off file `contexts.jsonl` |
| 9 | the checks |

The X3 **run record** goes to `results/<experiment>/runs/` next to the v1 runs. Every other file is in `results/<experiment>/x3/`: `contexts.jsonl`, `run_metrics.parquet`, `question_metrics.parquet`, `win500/metrics.{csv,xlsx}`, `comparison.csv`, `by_family.csv`, `paired.csv`, `evidence_losses.csv`, `cut_sweep.csv`, `keyword_rule.csv`, `diagrams/`.

Everything is cached and the run resumes from `partial.jsonl`. The first run makes the jina calls (all new: jina is cached per ordered candidate list); the filtered first-stage lists of the questions that the starting keyword lists detect are already in `data/cache/search.jsonl`. After the first run every step works from the caches: set `SERVICES = False` to run steps 2 and 4–9 with no Qdrant and no encoder (for example while vLLM is up). Editing `engine_keywords.toml` gives a new run id and a new set of jina calls; the old run stays.

X3's run record holds the windows the LLM would get (1–8), as the gate-on runs do, so its `@10` columns describe a short list. The columns to decide on are `section_hit@1`, `section_hit@3`, `evidence_recall@ctx` and the context tokens.

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # all paths below are relative to the repo root
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
DATA = ROOT / "data"
from csrag import experiment
EXP = experiment.active()  # `active` in experiments.toml, or the CSRAG_EXPERIMENT environment variable
RESULTS = ROOT / EXP.dir   # results/<experiment name>/: every result of this experiment
print(EXP.summary())

experiment 'run2' -> results/run2/  (Question set 1 + question set 2 (paraphrase, section_id, twin_topic, multi_section))
   100  data/eval/questions.jsonl
   177  data/eval/set2/paraphrase.jsonl
    50  data/eval/set2/section_id.jsonl
    60  data/eval/set2/twin_topic.jsonl
    90  data/eval/set2/multi_section.jsonl
   477  questions, 469 answerable | qrels: results/run2/qrels.jsonl


In [2]:
import json, tempfile, time
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from tqdm.auto import tqdm
from csrag import engine_rule, evaluate as ev, final, report as rp, runs
from csrag.corpus import COLLECTIONS, Corpus
from csrag.retrievers import Pipeline, Resources, grid, method_config, prompt_hashes
from csrag.retrievers.query_llm import vllm_healthy

PARAMS_VERSION = "v3"  # X3's parameters; the baselines are the stored v1 runs
INDEX = "win500"  # X3 runs on the windows index only
SERVICES = True  # False: no Qdrant and no encoder, everything from the caches (needs a completed X3 run)
ALLOW_CPU_RERANK = False  # True only if you accept hours of CPU reranking (use the GPU encoder)
ABLATIONS = []  # optional extra real runs, one more jina pass each: any of "no_filter", "rerank_input_50"
MAX_EXCLUDED_GOLD = 2  # check 6: questions whose gold section the keyword rule removes
TARGET_HIT1, TARGET_EVIDENCE = 0.90, 0.95  # check 2: overall Hit@1 and evidence in context
SWEEP_MIN_SCORES, SWEEP_MAX_KEEPS = [-0.05, 0.0, 0.05, 0.1], [5, 8, 10, 15]  # step 6
OUT = RESULTS / "x3"  # every file of the final pipeline except the run record, which is in runs/

assert set(ABLATIONS) <= set(final.ABLATION_LABELS), f"unknown ablation; defined: {sorted(final.ABLATION_LABELS)}"
corpus = Corpus(DATA)
assert corpus.has_windows, "run notebook 04 first"
if SERVICES:
    assert not vllm_healthy(), "vLLM is running: `docker compose stop vllm`, then `docker compose up -d qdrant && docker compose up -d --force-recreate encoder-gpu`"
    res = Resources.connect(corpus, DATA, with_llm=False)
    health = res.encoder.wait_ready()
    print("encoder on", health["device"], health.get("gpu") or "", "| dtype", health["dtype"])
    assert health["device"] == "cuda" or ALLOW_CPU_RERANK, (
        "the encoder runs on the CPU (hours of reranking): `docker compose stop encoder && docker compose up -d --force-recreate encoder-gpu`, "
        "or set ALLOW_CPU_RERANK = True")
    n = res.client.get_collection(COLLECTIONS[INDEX]).points_count
    assert n == len(corpus.units(INDEX)), f"{INDEX}: {n} points, run notebook 05"
    print("Qdrant collection", COLLECTIONS[INDEX], "ready")  # no ColBERT check: X3 does not use ColBERT
else:
    res = Resources.connect(corpus, DATA, with_llm=False, with_encoder=False)
    print("SERVICES = False: working from the caches only")

/home/goksu/projects/ikincitik/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:287: UserWarning: Qdrant client version 1.20.0 is incompatible with server version 1.18.3. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


encoder on cuda NVIDIA GeForce RTX 3090 | dtype float32
Qdrant collection cse_win500_v1 ready


## 1. Questions, keyword rules, X3 config and baselines

The baselines are looked up by their config in this experiment's `runs/`: X2 (BM25 + dense → RRF k = 60 → 50 candidates → jina → top 20, gate off), X2 with gate B, and H1 (the hybrid before the reranker), all v1 on windows.

In [3]:
questions, problems = ev.load_questions(EXP.questions, corpus)
assert not any(v for k, v in problems.items() if k != "quote_ambiguous"), "question validation failed: run notebook 06"
qrels = ev.read_qrels(EXP.qrels_path)
answerable = [q for q in questions if q["answerable"]]
assert {r["question_id"] for r in qrels} == {q["question_id"] for q in answerable}, "qrels do not match the questions: run notebook 06"
families = list(dict.fromkeys(ev.family_of(q) for q in answerable))
prompts = prompt_hashes()  # only for check 1 (the v1 grid)

rules = engine_rule.load()  # engine_keywords.toml, validated and normalised
rules_hash = engine_rule.rules_hash(rules)
x3_cfg = method_config("X3", INDEX, False, PARAMS_VERSION, engine_rules_hash=rules_hash)
X3_ID = runs.run_id_of(x3_cfg)
X2_ID = final.find_run("X2", INDEX, False, "v1")
X2_ON_ID = final.find_run("X2", INDEX, True, "v1")
H1_ID = final.find_run("H1", INDEX, False, "v1")
BASELINES = [X2_ID, X2_ON_ID, H1_ID]
stale = [r for r in BASELINES if not runs.same_questions(r, questions)]
assert not stale, f"baseline runs made with another question set: {stale}"

print(f"{len(questions)} questions, {len(answerable)} answerable | families: {families}")
print("X3 run id:", X3_ID, "| keyword rules:", rules_hash)
print("baselines:", BASELINES)
print(json.dumps({k: x3_cfg[k] for k in ("first_stages", "candidates", "fusion", "filter", "rerank", "cut", "k")}, indent=1))

477 questions, 469 answerable | families: ['set1', 'paraphrase', 'section_id', 'twin_topic', 'multi_section']
X3 run id: X3-win500-gateoff-v3-7d629b5d | keyword rules: 3a156fd8
baselines: ['X2-win500-gateoff-v1-f352b4b8', 'X2-win500-gateon-v1-8b1df080', 'H1-win500-gateoff-v1-9d5dfb15']
{
 "first_stages": [
  "bm25",
  "dense"
 ],
 "candidates": 50,
 "fusion": {
  "type": "rrf",
  "k": 10
 },
 "filter": {
  "type": "keywords",
  "field": "engine_type",
  "rules_hash": "3a156fd8"
 },
 "rerank": {
  "model": "jina",
  "input": 30,
  "top": 8
 },
 "cut": {
  "min_score": 0.0,
  "min_keep": 1,
  "max_keep": 8
 },
 "k": 8
}


## 2. The keyword rule against the labels

The rule reads the question text only. The labels (`engine_type` of each question) are used here to see how the rule performs, and for nothing else. A wrong filter removes the gold section completely: it is the only way X3 can do worse than X2 on a question, so the questions whose gold section the rule would remove are listed, and the notebook stops here if there are more than `MAX_EXCLUDED_GOLD`, before any reranker time is spent. Edit `engine_keywords.toml`, then re-run steps 1 and 2 (the rules are read in step 1).

In [4]:
print(json.dumps(rules, indent=1, ensure_ascii=False))
kr = final.keyword_report(questions, rules, corpus, INDEX)
by_label, reasons = final.keyword_summary(kr)
display(Markdown("**Label (rows) against detected type (columns)**"))
display(by_label)
display(Markdown("**Detection reasons**"))
display(reasons.rename("questions").to_frame().T)
excluded = kr[kr["n_excluded_gold"] > 0]
display(Markdown(f"**{len(excluded)} question(s) whose gold section the rule would remove** (limit {MAX_EXCLUDED_GOLD})"))
display(excluded[["question_id", "family", "label", "detected", "matched", "excluded_gold", "question"]])
OUT.mkdir(parents=True, exist_ok=True)
kr.to_csv(OUT / "keyword_rule.csv", index=False, encoding="utf-8")
assert len(excluded) <= MAX_EXCLUDED_GOLD, (
    "the keyword rule removes too many gold sections: edit engine_keywords.toml and re-run steps 1 and 2 "
    "(or raise MAX_EXCLUDED_GOLD) before the run in step 3")

{
 "engine_types": {
  "turbine": [
   "gas turbine",
   "jet",
   "jet engine",
   "ramjet",
   "turbine",
   "turbine engine",
   "turbine powered",
   "turbo shaft",
   "turbofan",
   "turbojet",
   "turboprop",
   "turboshaft"
  ],
  "piston": [
   "compression ignition",
   "crankshaft",
   "cylinder",
   "diesel",
   "piston",
   "piston engine",
   "reciprocating",
   "spark ignition"
  ]
 },
 "ignore": [
  "turbocharger turbine"
 ]
}


**Label (rows) against detected type (columns)**

detected,none,piston,turbine,All
label,,,,
all,95,6,26,127
piston,13,105,0,118
turbine,74,0,158,232
All,182,111,184,477


**Detection reasons**

reason,one_type,no_cue,both_types
questions,295,162,20


**0 question(s) whose gold section the rule would remove** (limit 2)

,question_id,family,label,detected,matched,excluded_gold,question


## 3. Run X3

`runs.execute` is resumable (`partial.jsonl`) and uses the caches; the jina calls are the new work. The record goes to `results/<experiment>/runs/`. The optional ablations (`ABLATIONS`) are separate run records.

In [5]:
cfgs = {"X3": x3_cfg, **{a: final.ablation_config(a, INDEX, PARAMS_VERSION, rules_hash) for a in ABLATIONS}}
run_ids = {name: runs.run_id_of(cfg) for name, cfg in cfgs.items()}
for name, cfg in cfgs.items():
    rid = run_ids[name]
    assert SERVICES or runs.is_complete(rid), f"{rid} is not complete: run the notebook once with SERVICES = True"
    if SERVICES:
        assert not vllm_healthy(), "vLLM is running: `docker compose stop vllm` before reranking"
    t0 = time.time()
    runs.execute(cfg, Pipeline(cfg, res, engine_rules=rules).run, questions, res.meta(cfg["index"]), progress=tqdm)
    print(rid, "|", round(time.time() - t0), "s")
assert run_ids["X3"] == X3_ID

X3-win500-gateoff-v3-7d629b5d:   0%|          | 0/477 [00:00<?, ?it/s]

X3-win500-gateoff-v3-7d629b5d | 1550 s


## 4. The standard metric files

The same files notebook 09 writes (`run_metrics.parquet`, `question_metrics.parquet`, `win500/metrics.{csv,xlsx}` with all sheets), for X3 and the three baselines, into `results/<experiment>/x3/`. The code is notebook 09's (`ev.evaluate_all`, `rp.write_index_files`), so every metric, CI and breakdown is computed as for every other run. The `significance` sheet stays empty: the paired comparisons are in step 5. Re-running notebook 09 afterwards would add X3 as one more row to the main tables with the same values.

In [8]:
metric_ids = list(dict.fromkeys([X3_ID, *[run_ids[a] for a in ABLATIONS], *BASELINES]))
w = final.write_metric_files(metric_ids, questions, qrels, corpus, OUT, INDEX, progress=tqdm)
table_csv = pd.read_csv(OUT / INDEX / "metrics.csv", encoding="utf-8")
COLS = ["method", "gate", "params_version", "filter", "rerank", "section_hit@1", "section_hit@3", "section_ndcg@10", "section_recall@10",
        "evidence_recall@10", "evidence_recall@ctx", "context_tokens", "n_hits", "latency_rerank_s", "latency_total_s"]
display(table_csv.set_index("run_id")[[c for c in COLS if c in table_csv.columns]].round(3))
diagram = rp.write_diagram(X3_ID, OUT / "diagrams", rm=w["run_metrics"], sig=None, gates=w["gates"], costs=w["costs"])
display(Markdown(diagram.read_text(encoding="utf-8")))

evaluate:   0%|          | 0/4 [00:00<?, ?it/s]

,method,gate,params_version,filter,rerank,section_hit@1,section_hit@3,section_ndcg@10,section_recall@10,evidence_recall@10,evidence_recall@ctx,context_tokens,n_hits,latency_rerank_s,latency_total_s
run_id,,,,,,,,,,,,,,,
X3-win500-gateoff-v3-7d629b5d,X3,False,v3,engine_type (keywords),"jina-reranker-v3 (top 30), cut score ≥ 0.0 kee...",0.917,0.979,0.865,0.979,0.975,0.975,1624.547,4.350,3.245,3.265
X2-win500-gateoff-v1-f352b4b8,X2,False,v1,-,jina-reranker-v3 (top 50),0.853,0.970,0.864,0.986,0.981,0.984,7634.855,20.000,8.208,8.230
X2-win500-gateon-v1-8b1df080,X2,True,v1,-,jina-reranker-v3 (top 50),0.846,0.947,0.814,0.932,0.920,0.920,2459.943,6.545,0.691,0.714
H1-win500-gateoff-v1-9d5dfb15,H1,False,v1,-,-,0.646,0.842,0.739,0.928,0.886,0.950,7983.430,20.000,0.000,0.022


# X3-win500-gateoff-v3-7d629b5d

X3: H1 + keyword engine-type filter -> jina-reranker-v3 -> score cut · index `win500` · gate off · params `v3` · config `7d629b5dc623`

```mermaid
flowchart LR
  Q(["question"])
  FL{"filter<br/>engine_type ∈ {x, all}<br/>keyword list"}
  Q --> FL
  IX[("cse_win500_v1<br/>500-token windows")]
  FL --> IX
  S0["BM25<br/>top-50"]
  IX --> S0
  S1["bge-m3 dense<br/>top-50"]
  IX --> S1
  FU["RRF<br/>k = 10"]
  S0 --> FU
  S1 --> FU
  RR["X3 jina-reranker-v3<br/>top 30 in"]
  FU --> RR
  CT{"score cut<br/>score ≥ 0.0<br/>keep 1–8"}
  RR --> CT
  CTX(["final context<br/>k ≤ 8"])
  CT --> CTX
  subgraph M["metrics"]
    MP["section nDCG@10 0.865 [0.848, 0.881]<br/>Recall@10 0.979<br/>evidence recall@10 0.975<br/>latency 3.27 s/question<br/>n = 469 answerable"]
  end
  CTX -.- MP
```


## 5. X2 vs X3

| row | what it isolates |
|---|---|
| X2, 20 returned | the baseline as reported |
| X2, top 8 | a fixed cut of the same size limit (replay of the stored list) |
| X2 + X3's cut-off | the cut-off alone (replay of the stored list) |
| X2, gate on | the old way to shorten the context |
| X3, no cut-off (30 windows) | what the filter and the 30 candidates give before the cut (replay of `stats["reranked"]`); its nDCG@10 and Recall@10 can be set against X2's 20 |
| **X3** | everything |

Section level; 95% bootstrap CIs on the same resamples for every row. Reranker and total seconds are those of the run that made the jina call (a replay row shares the call of its source run). Context precision is the share of returned windows that lie in a gold section. Differences are paired by question; `p_holm` corrects across the six metrics within each pair. As noted in the README, pooled CIs and tests treat the questions as independent although a paraphrase shares its gold with its original, so they are too narrow: read the families separately.

In [9]:
CUT = x3_cfg["cut"]
X2_LABEL, X2_ON_LABEL, X3_LABEL = "X2, 20 returned", "X2, gate on", "X3"
mo = lambda hits: final.metrics_of(hits, questions, qrels, corpus, INDEX)
rows = {
    X2_LABEL: mo(runs.load_hits(X2_ID)),
    "X2, top 8": mo(final.replay_cut(X2_ID, corpus, -np.inf, 8, 8)),
    "X2 + X3's cut-off": mo(final.replay_cut(X2_ID, corpus, **CUT)),
    X2_ON_LABEL: mo(runs.load_hits(X2_ON_ID)),
    "X3, no cut-off (30 windows)": mo(final.replay_cut(X3_ID, corpus, **final.UNCUT)),
    X3_LABEL: mo(runs.load_hits(X3_ID)),
    **{final.ABLATION_LABELS[a]: mo(runs.load_hits(run_ids[a])) for a in ABLATIONS},
}
source = {X2_LABEL: X2_ID, "X2, top 8": X2_ID, "X2 + X3's cut-off": X2_ID, X2_ON_LABEL: X2_ON_ID, "X3, no cut-off (30 windows)": X3_ID,
          X3_LABEL: X3_ID, **{final.ABLATION_LABELS[a]: run_ids[a] for a in ABLATIONS}}
lat = {label: final.latency(rid) for label, rid in source.items()}
table, by_family, paired = final.compare(rows, [(X3_LABEL, X2_LABEL), (X3_LABEL, "X2 + X3's cut-off")], lat)
SHOW = ["row", "n", "hit@1", "hit@1_ci_lo", "hit@1_ci_hi", "hit@3", "hit@3_ci_lo", "hit@3_ci_hi", "mrr@10", "ndcg@10", "recall@10", "evidence_ctx",
        "windows_mean", "windows_median", "tokens_mean", "precision", "rerank_s", "total_s"]
display(table[[c for c in SHOW if c in table.columns]].set_index("row").round(3))
table.to_csv(OUT / "comparison.csv", index=False, encoding="utf-8")
by_family.to_csv(OUT / "by_family.csv", index=False, encoding="utf-8")
paired.to_csv(OUT / "paired.csv", index=False, encoding="utf-8")

,n,hit@1,hit@1_ci_lo,hit@1_ci_hi,hit@3,hit@3_ci_lo,hit@3_ci_hi,mrr@10,ndcg@10,recall@10,evidence_ctx,windows_mean,windows_median,tokens_mean,precision,rerank_s,total_s
row,,,,,,,,,,,,,,,,,
"X2, 20 returned",469,0.853,0.821,0.885,0.970,0.953,0.985,0.912,0.864,0.986,0.984,20.000,20.0,7631.574,0.108,8.208,8.230
"X2, top 8",469,0.853,0.821,0.885,0.970,0.953,0.985,0.912,0.853,0.982,0.980,8.000,8.0,3001.422,0.223,8.208,8.230
X2 + X3's cut-off,469,0.853,0.821,0.885,0.970,0.953,0.985,0.911,0.839,0.978,0.976,5.171,5.0,1917.861,0.392,8.208,8.230
"X2, gate on",469,0.846,0.814,0.878,0.947,0.925,0.966,0.895,0.814,0.932,0.920,6.544,6.0,2459.433,0.246,0.691,0.714
"X3, no cut-off (30 windows)",469,0.917,0.891,0.940,0.979,0.964,0.991,0.949,0.894,0.986,0.984,30.000,30.0,11903.680,0.074,3.245,3.265
X3,469,0.917,0.891,0.940,0.979,0.964,0.991,0.949,0.865,0.979,0.975,4.380,4.0,1638.902,0.471,3.245,3.265


In [10]:
for metric in ("hit@1", "evidence_ctx"):
    display(Markdown(f"**{metric} by family**"))
    display(by_family.pivot(index="row", columns="family", values=metric).reindex(index=list(rows), columns=families).round(3))
display(Markdown("**twin_topic: questions with a section of the other engine type at rank 1**"))
display(pd.DataFrame({X2_LABEL: final.other_type_at_rank1(runs.load_hits(X2_ID), questions, corpus, INDEX),
                      X3_LABEL: final.other_type_at_rank1(runs.load_hits(X3_ID), questions, corpus, INDEX)}).T)
display(Markdown("**Paired differences (a − b)**"))
display(paired.round(4))

**hit@1 by family**

family,set1,paraphrase,section_id,twin_topic,multi_section
row,,,,,
"X2, 20 returned",0.946,0.859,0.94,0.517,0.922
"X2, top 8",0.946,0.859,0.94,0.517,0.922
X2 + X3's cut-off,0.946,0.859,0.94,0.517,0.922
"X2, gate on",0.957,0.836,0.96,0.500,0.922
"X3, no cut-off (30 windows)",0.978,0.876,0.96,0.900,0.922
X3,0.978,0.876,0.96,0.900,0.922


**evidence_ctx by family**

family,set1,paraphrase,section_id,twin_topic,multi_section
row,,,,,
"X2, 20 returned",1.0,0.972,1.00,1.000,0.974
"X2, top 8",1.0,0.972,1.00,1.000,0.952
X2 + X3's cut-off,1.0,0.960,1.00,1.000,0.952
"X2, gate on",1.0,0.898,0.98,0.917,0.849
"X3, no cut-off (30 windows)",1.0,0.972,1.00,1.000,0.974
X3,1.0,0.966,0.99,1.000,0.943


**twin_topic: questions with a section of the other engine type at rank 1**

,family,n,other_type_at_rank1,share
"X2, 20 returned",twin_topic,52,25,0.480769
X3,twin_topic,52,0,0.0


**Paired differences (a − b)**

,a,b,metric,mean_a,mean_b,diff,diff_ci_lo,diff_ci_hi,p,p_holm
0,X3,"X2, 20 returned",hit@1,0.9168,0.8529,0.0640,0.0384,0.0896,0.0001,0.0006
1,X3,"X2, 20 returned",hit@3,0.9787,0.9701,0.0085,-0.0043,0.0213,0.3494,0.7268
2,X3,"X2, 20 returned",mrr@10,0.9490,0.9120,0.0370,0.0216,0.0532,0.0001,0.0006
3,X3,"X2, 20 returned",ndcg@10,0.8649,0.8644,0.0006,-0.0123,0.0138,0.9339,0.9339
4,X3,"X2, 20 returned",recall@10,0.9787,0.9861,-0.0075,-0.0181,0.0032,0.2423,0.7268
5,X3,"X2, 20 returned",evidence_ctx,0.9751,0.9844,-0.0092,-0.0199,0.0014,0.1271,0.5083
6,X3,X2 + X3's cut-off,hit@1,0.9168,0.8529,0.0640,0.0384,0.0896,0.0001,0.0006
7,X3,X2 + X3's cut-off,hit@3,0.9787,0.9701,0.0085,-0.0043,0.0213,0.3494,1.0000
8,X3,X2 + X3's cut-off,mrr@10,0.9490,0.9112,0.0377,0.0222,0.0539,0.0001,0.0006
9,X3,X2 + X3's cut-off,ndcg@10,0.8649,0.8393,0.0256,0.0138,0.0379,0.0001,0.0006


In [11]:
loss = final.evidence_losses(X3_ID, runs.load_hits(X2_ID), questions, qrels, corpus)
loss.to_csv(OUT / "evidence_losses.csv", index=False, encoding="utf-8")
display(Markdown(f"**{len(loss)} question(s) where X3's context has less evidence than X2's 20 windows**, by cause "
                 "(`filter`: the keyword rule removed a gold section; `candidates`: not among the 30 sent to jina; `cut`: dropped by the cut-off)"))
display(loss["cause"].value_counts().rename("questions").to_frame().T)
display(loss.drop(columns="question").round(3))

**10 question(s) where X3's context has less evidence than X2's 20 windows**, by cause (`filter`: the keyword rule removed a gold section; `candidates`: not among the 30 sent to jina; `cut`: dropped by the cut-off)

cause,cut,candidates
questions,7,3


,question_id,family,evidence_x2,evidence_x3,evidence_in_candidates,detected,cause
0,pa019u,paraphrase,1.0,0.000,0.0,NaN,candidates
1,pa056h,paraphrase,1.0,0.000,0.0,turbine,candidates
2,pa066h,paraphrase,1.0,0.000,1.0,NaN,cut
3,id016,section_id,1.0,0.500,1.0,NaN,cut
4,ms038,multi_section,1.0,0.667,1.0,turbine,cut
5,ms043,multi_section,1.0,0.500,0.5,NaN,candidates
6,ms052,multi_section,1.0,0.500,1.0,NaN,cut
7,ms072,multi_section,1.0,0.500,1.0,NaN,cut
8,ms079,multi_section,1.0,0.500,1.0,NaN,cut
9,ms089,multi_section,1.0,0.500,1.0,NaN,cut


## 6. Other cut-offs on X3's own reranked lists

`stats["reranked"]` holds every candidate with its jina score, so other thresholds cost no reranker call. `cut_loss` is the share of questions whose evidence in context is lower than that of the whole uncut ordering. The second table is the configured cut-off by family.

In [12]:
sweep = final.cut_sweep(X3_ID, questions, qrels, corpus, SWEEP_MIN_SCORES, SWEEP_MAX_KEEPS, min_keep=CUT["min_keep"])
sweep.to_csv(OUT / "cut_sweep.csv", index=False, encoding="utf-8")
display(sweep[sweep["family"] == "all"].drop(columns="family").round(3))
display(Markdown(f"**The configured cut-off (score ≥ {CUT['min_score']}, keep {CUT['min_keep']}–{CUT['max_keep']}) by family**"))
cols = ["family", "n", "hit@1", "hit@3", "evidence_ctx", "windows_mean", "tokens_mean", "cut_loss"]
display(sweep[(sweep["min_score"] == CUT["min_score"]) & (sweep["max_keep"] == CUT["max_keep"])][cols].round(3))

,min_score,max_keep,n,hit@1,hit@3,evidence_ctx,windows_mean,tokens_mean,cut_loss
0,-0.05,5,469,0.917,0.979,0.966,4.501,1673.580,0.030
6,-0.05,8,469,0.917,0.979,0.979,6.264,2385.834,0.011
12,-0.05,10,469,0.917,0.979,0.981,7.132,2732.288,0.006
18,-0.05,15,469,0.917,0.979,0.983,8.448,3266.009,0.002
24,0.00,5,469,0.917,0.979,0.963,3.561,1302.708,0.034
30,0.00,8,469,0.917,0.979,0.975,4.380,1638.902,0.017
36,0.00,10,469,0.917,0.979,0.977,4.676,1753.294,0.013
42,0.00,15,469,0.917,0.979,0.978,5.062,1909.906,0.011
48,0.05,5,469,0.917,0.979,0.963,2.825,1009.277,0.034
54,0.05,8,469,0.917,0.979,0.974,3.162,1148.529,0.019


**The configured cut-off (score ≥ 0.0, keep 1–8) by family**

,family,n,hit@1,hit@3,evidence_ctx,windows_mean,tokens_mean,cut_loss
30,all,469,0.917,0.979,0.975,4.380,1638.902,0.017
31,set1,92,0.978,1.000,1.000,3.772,1379.978,0.000
32,paraphrase,177,0.876,0.949,0.966,4.249,1587.667,0.006
33,section_id,50,0.960,1.000,0.990,3.680,1346.060,0.020
34,twin_topic,60,0.900,1.000,1.000,4.400,1678.167,0.000
35,multi_section,90,0.922,0.989,0.943,5.633,2140.856,0.067


## 7. Unanswerable questions (inspection only, not scored)

X3 always returns at least one window. The scores show how a reader (or the LLM) could tell that nothing answers the question.

In [13]:
un = final.unanswerable_view(X3_ID, questions, corpus)
display(un.round(3) if len(un) else "no unanswerable questions in this experiment")

,question_id,question,detected,windows,top1_score,sections
0,q093,"What expansion space, as a percentage of capac...",NaN,1,0.403,AMC E 570
1,q094,How many successful starts must an auxiliary p...,NaN,2,0.314,CS-E 370 | CS-E 750
2,q095,How many times must a helicopter main rotor br...,NaN,5,0.307,CS-E 900 | CS-E 740 | CS-E 710 | CS-E 180
3,q096,What maximum in-flight shutdown (IFSD) rate pe...,NaN,1,0.472,CS-E 1040
4,q097,What fuel contamination rate applies when test...,piston,3,0.294,AMC E 470 | AMC E 150(a) | AMC E 30
5,q098,Within how many seconds must the flight crew b...,NaN,3,0.355,AMC E 710 | AMC E 910 | CS-E 710
6,q099,Which subparts of the engine certification spe...,NaN,4,0.183,CS-E 10 | CS-E 175 | AMC E 20 | CS-E 1000
7,q100,What is the maximum permissible coolant temper...,piston,2,0.171,CS-E 300 | AMC E 40(d)


## 8. The hand-off file

`results/<experiment>/x3/contexts.jsonl`, one line per question. `hits` is what X3 hands over (1–8 windows with their text) and goes into the prompt; `reranked` is every candidate with its jina score, best first, without text (the text of a window is `corpus.units("win500")[unit_id]["text"]`). A generation notebook calls `final.read_contexts(path, question_ids=[...])` with 3–5 ids and needs only vLLM. Gold answers stay in the question files and are joined by `question_id`.

In [14]:
contexts_path = final.write_contexts(X3_ID, questions, corpus, OUT / "contexts.jsonl")
print(contexts_path, "|", round(contexts_path.stat().st_size / 1e6, 1), "MB |", len(final.read_contexts(contexts_path)), "questions")
for r in final.read_contexts(contexts_path, [q["question_id"] for q in answerable[:3]]):
    print("=" * 100)
    print(r["question_id"], f"({r['family']}) detected: {r['engine_type_detected']} | {len(r['hits'])} windows, {r['context_tokens']} tokens, {len(r['reranked'])} reranked")
    print(r["question"])
    for h in r["hits"]:
        print(f"  {h['rank']}. {h['score']:+.3f}  {h['unit_id']}  (p. {h['page_start']}, {h['n_tokens']} tokens)  {h['text'][:110]!r}")

/home/goksu/projects/ikincitik/results/run2/x3/contexts.jsonl | 4.7 MB | 477 questions
q001 (set1) detected: None | 4 windows, 1171 tokens, 30 reranked
Besides the acceptable means of compliance in Book 2 of the engine certification specifications, which other document may also provide acceptable means of compliance to the Book 1 specifications?
  1. +0.591  AMC General (SUBPART A)#w1  (p. 16, 41 tokens)  'AMC General (SUBPART A) General\n\nIn addition to the acceptable means of compliance in Book 2 of these Certific'
  2. +0.056  AMC E 80#w1  (p. 46, 489 tokens)  'AMC E 80 Equipment (part 1/9)\n\n(1) The need for additional specifications in the equipment specifications shou'
  3. +0.029  AMC E 790(a)(2)#w17  (p. 202, 203 tokens)  'AMC E 790(a)(2) Rain and Hail Ingestion – Turbine Engine Power/ Thrust Loss and Instability in Extreme Conditi'
  4. +0.027  AMC E 910#w1  (p. 250, 364 tokens)  'AMC E 910 Relighting In Flight\n\n(1) AMC 25.903(e)(2) contains guidance that can be used to es

## 9. Checks

1. **Existing run ids are unchanged:** the ids of `grid("v1")` are the stored v1 runs, all complete.
2. **Targets:** overall Hit@1 and evidence in context at or above the targets; Hit@3 not below "X2, top 8". A failure names the families.
3. **No label leakage:** the X3 pipeline holds no labels, its filter is the keyword rule, and `retrieve` receives the question text only.
4. **Cut-off:** every list between `min_keep` and `max_keep`, scores descending, a prefix of `stats["reranked"]`, the cut as long as the rule allows.
5. **Filter:** every candidate has `engine_type ∈ {detected type, all}`; the candidates are the fused top 30 of the cached first-stage lists (unfiltered when no type was detected).
6. **Keyword rule:** the gold sections it removes stay within `MAX_EXCLUDED_GOLD`.
7. **Reproducibility:** a second run from the caches, written to a temporary root, gives identical hits.
8. **Hand-off:** `contexts.jsonl` matches `hits.parquet`, `stats.jsonl` and the corpus.
9. **Metric files:** one row per evaluated run; the baseline rows equal notebook 09's rows in `results/<experiment>/win500/metrics.csv` (when it exists); X3's `section_hit@1` and `section_hit@3` equal the comparison table.

In [16]:
import inspect

p3 = Pipeline(x3_cfg, res, engine_rules=rules)


def lists_for(text, flt):
    """The cached first-stage lists of X3's two signals for a question and a Qdrant filter."""
    q = p3._query(text)
    return [[(u, s) for u, s in p3.searcher.search(INDEX, sig, q, flt, x3_cfg["candidates"])["hits"]] for sig in x3_cfg["first_stages"]]


def problem(condition, message):
    """[message] when the condition holds, else []. (`list * bool` fails for a numpy bool.)"""
    return [message] if bool(condition) else []


grid_ids = {runs.run_id_of(c) for c in grid("v1", prompts=prompts)}
stored_v1 = set(runs.list_runs(params_version="v1"))
with tempfile.TemporaryDirectory() as tmp:
    runs.execute(x3_cfg, Pipeline(x3_cfg, res, engine_rules=rules).run, questions, res.meta(INDEX), root=tmp)
    reproduced = runs.load_hits(X3_ID, tmp).equals(runs.load_hits(X3_ID))
t_csv, t_cmp = table_csv.set_index("run_id"), table.set_index("row")

checks = {
    "1 existing run ids unchanged": (
        problem(grid_ids - stored_v1, f"in the v1 grid but not stored: {sorted(grid_ids - stored_v1)[:3]}")
        + problem(stored_v1 - grid_ids, f"stored v1 runs outside the grid: {sorted(stored_v1 - grid_ids)[:3]}")
        + [f"{r} is not complete" for r in sorted(grid_ids) if not runs.is_complete(r)]),
    "2 targets": final.check_targets(rows, X3_LABEL, "X2, top 8", TARGET_HIT1, TARGET_EVIDENCE),
    "3 no label leakage": (
        problem(p3.labels is not None, "the X3 pipeline holds labels")
        + problem(x3_cfg["filter"]["type"] != "keywords", "the X3 filter is not the keyword rule")
        + problem(list(inspect.signature(p3.retrieve).parameters) != ["question_id", "text", "k"], "retrieve takes more than the question text")),
    "4 cut-off": final.check_cut(X3_ID),
    "5 filter": final.check_filter(X3_ID, questions, corpus, lists_for),
    "6 keyword rule": problem(len(excluded) > MAX_EXCLUDED_GOLD, f"{len(excluded)} questions lose a gold section (limit {MAX_EXCLUDED_GOLD})"),
    "7 reproducibility": problem(not reproduced, "a second run gives other hits"),
    "8 hand-off": final.check_contexts(contexts_path, X3_ID, questions, corpus),
    "9 metric files": (
        final.check_metric_files(OUT, metric_ids, BASELINES, RESULTS, INDEX)
        + problem(abs(t_csv.loc[X3_ID, "section_hit@1"] - t_cmp.loc[X3_LABEL, "hit@1"]) > 1e-9, "section_hit@1 differs from the comparison table")
        + problem(abs(t_csv.loc[X3_ID, "section_hit@3"] - t_cmp.loc[X3_LABEL, "hit@3"]) > 1e-9, "section_hit@3 differs from the comparison table")),
}
for name, probs in checks.items():
    print("PASS" if not probs else "FAIL", name, "" if not probs else f"({len(probs)}): {probs[:3]}")
assert not any(checks.values()), "checks failed: see above"
print("all checks passed")

PASS 1 existing run ids unchanged 
PASS 2 targets 
PASS 3 no label leakage 
PASS 4 cut-off 
PASS 5 filter 
PASS 6 keyword rule 
PASS 7 reproducibility 
PASS 8 hand-off 
PASS 9 metric files 
all checks passed


## Next: the generation notebooks

```bash
docker compose stop encoder-gpu && docker compose up -d vllm      # VLLM_CONFIG=serve.retrieval.yaml (.env); ~2.5 min to healthy
docker compose logs -f vllm
```

Read three to five questions from `results/<experiment>/x3/contexts.jsonl` with `final.read_contexts(path, question_ids=[...])` and try other models and prompts on their `hits`. Qdrant and the encoder are not needed.